# M3 profile: AIS + wave data (M2 tables, read-only)
Profiles every table and column the project question depends on **before** any cleaning decision.
Reads only the M2 tables (`PRAGMA query_only = ON`). Run top to bottom on a fresh `python src/load.py` build.

In [1]:
import sqlite3, statistics, sys, unicodedata, math
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "schema.sql").exists())
sys.path.insert(0, str(ROOT / "src"))
from db import connect, DEFAULT_DB
con = connect()
con.execute("PRAGMA query_only = ON")
print("database:", DEFAULT_DB)

def show(sql, params=()):
    cur = con.execute(sql, params)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    w = [max(len(str(x)) for x in [c] + [r[i] for r in rows]) for i, c in enumerate(cols)]
    print("  ".join(c.ljust(w[i]) for i, c in enumerate(cols)))
    print("  ".join("-" * x for x in w))
    for r in rows:
        print("  ".join(str(v).ljust(w[i]) for i, v in enumerate(r)))
    if not rows:
        print("(no rows)")

database: /home/it/turkish-straits-maritime-pipeline/data/maritime.db


## 1. Row counts, and rows per entity
Unequal coverage makes comparisons meaningless.

In [2]:
show("SELECT 'vessel' AS tbl, COUNT(*) AS n FROM vessel UNION ALL SELECT 'marine_point', COUNT(*) FROM marine_point "
     "UNION ALL SELECT 'ais_position', COUNT(*) FROM ais_position UNION ALL SELECT 'wave_reading', COUNT(*) FROM wave_reading "
     "UNION ALL SELECT 'quarantine', COUNT(*) FROM quarantine")
print("\nAIS positions per vessel (how many vessels have k positions):")
show("SELECT k AS positions_per_vessel, COUNT(*) AS vessels FROM (SELECT COUNT(*) AS k FROM ais_position GROUP BY vessel_key) GROUP BY k ORDER BY k")
print("\nwave readings per marine point:")
show("SELECT p.point_key, p.latitude, p.longitude, COUNT(w.observed_at) AS readings, MIN(w.observed_at) AS first, MAX(w.observed_at) AS last "
     "FROM marine_point p LEFT JOIN wave_reading w USING (point_key) GROUP BY p.point_key")

tbl           n  
------------  ---
vessel        67 
marine_point  8  
ais_position  72 
wave_reading  960
quarantine    0  

AIS positions per vessel (how many vessels have k positions):
positions_per_vessel  vessels
--------------------  -------
1                     62     
2                     5      

wave readings per marine point:
point_key  latitude   longitude  readings  first              last             
---------  ---------  ---------  --------  -----------------  -----------------
1          38.458336  25.875015  120       2026-10-09T00:00Z  2026-10-22T23:00Z
2          40.041664  26.125015  120       2026-10-09T00:00Z  2026-10-22T23:00Z
3          40.458336  26.708344  120       2026-10-09T00:00Z  2026-10-22T23:00Z
4          40.625     26.958344  120       2026-10-09T00:00Z  2026-10-22T23:00Z
5          40.791664  27.958344  120       2026-10-09T00:00Z  2026-10-22T23:00Z
6          40.791664  29.291672  120       2026-10-09T00:00Z  2026-10-22T23:00Z
7          40.9583

## 2. Missing values: how many, which entity, which period

In [3]:
show("SELECT COUNT(*) AS n, SUM(latitude IS NULL) AS lat, SUM(longitude IS NULL) AS lon, SUM(sog IS NULL) AS sog, SUM(cog IS NULL) AS cog, "
     "SUM(true_heading IS NULL) AS heading, SUM(nav_status IS NULL) AS nav, SUM(rate_of_turn IS NULL) AS rot FROM ais_position")
print("\nwave_reading NULLs per point (a point that is NULL everywhere has no model coverage):")
show("SELECT point_key, COUNT(*) AS n, SUM(wave_height IS NULL) AS height, SUM(wave_direction IS NULL) AS direction, "
     "SUM(wave_period IS NULL) AS period, SUM(wind_wave_height IS NULL) AS wind FROM wave_reading GROUP BY point_key")
print("\nwave_reading NULL share of wave_height per UTC day:")
show("SELECT substr(observed_at, 1, 10) AS day, COUNT(*) AS n, ROUND(AVG(wave_height IS NULL), 3) AS null_share FROM wave_reading GROUP BY day ORDER BY day")
print("\nAIS: do the NULLs cluster in one vessel or hour? (missing sog by UTC hour)")
show("SELECT substr(observed_at, 1, 13) AS hour, COUNT(*) AS n, SUM(sog IS NULL) AS missing_sog FROM ais_position GROUP BY hour ORDER BY hour")

n   lat  lon  sog  cog  heading  nav  rot
--  ---  ---  ---  ---  -------  ---  ---
72  0    0    0    0    0        0    0  

wave_reading NULLs per point (a point that is NULL everywhere has no model coverage):
point_key  n    height  direction  period  wind
---------  ---  ------  ---------  ------  ----
1          120  0       0          0       0   
2          120  0       0          0       0   
3          120  0       0          0       0   
4          120  0       0          0       0   
5          120  0       0          0       0   
6          120  0       0          0       0   
7          120  0       0          0       0   
8          120  0       0          0       0   

wave_reading NULL share of wave_height per UTC day:
day         n    null_share
----------  ---  ----------
2026-10-09  192  0.0       
2026-10-10  192  0.0       
2026-10-20  192  0.0       
2026-10-21  192  0.0       
2026-10-22  192  0.0       

AIS: do the NULLs cluster in one vessel or hour? (missing

## 3. Min / median / max and most frequent values (sentinels hide in the extremes and the modes)

In [4]:
def summary(table, cols, key=None):
    for c in cols:
        vals = [v for (v,) in con.execute("SELECT " + c + " FROM " + table + " WHERE " + c + " IS NOT NULL")]
        if not vals:
            print("%-18s all NULL" % c); continue
        top = Counter(vals).most_common(4)
        print("%-18s n=%-5d min=%-9s median=%-9s max=%-9s top=%s" % (c, len(vals), min(vals), statistics.median(vals), max(vals), top))
summary("ais_position", ["latitude", "longitude", "sog", "cog", "true_heading", "nav_status", "rate_of_turn"])
print()
summary("wave_reading", ["wave_height", "wave_direction", "wave_period", "wind_wave_height"])
print("\nexact sentinel candidates in ais_position (count of rows):")
show("SELECT SUM(sog = 102.3) AS sog_102_3, SUM(cog = 360) AS cog_360, SUM(true_heading = 511) AS heading_511, SUM(nav_status = 15) AS nav_15, "
     "SUM(rate_of_turn = -128) AS rot_m128, SUM(latitude = 91) AS lat_91, SUM(longitude = 181) AS lon_181, "
     "SUM(latitude = 0 AND longitude = 0) AS null_island, SUM(sog = 0) AS sog_zero FROM ais_position")
print("\nimpossible-value rules (family 2):")
show("SELECT SUM(sog < 0) AS neg_sog, SUM(latitude NOT BETWEEN -90 AND 90) AS lat_range, SUM(longitude NOT BETWEEN -180 AND 180) AS lon_range, "
     "SUM(latitude NOT BETWEEN 35.0 AND 42.5 OR longitude NOT BETWEEN 25.0 AND 36.0) AS outside_bbox FROM ais_position")
show("SELECT SUM(wave_height < 0) AS neg_height, SUM(wave_height > 25) AS height_over_25, SUM(wave_direction NOT BETWEEN 0 AND 360) AS dir_range, "
     "SUM(wave_period < 0) AS neg_period, SUM(wind_wave_height > wave_height + 0.05) AS wind_over_total FROM wave_reading")

latitude           n=72    min=35.03510000000001 median=36.82708  max=42.48638666666667 top=[(37.464666666666666, 2), (42.48638666666667, 1), (40.727105, 1), (37.01352166666666, 1)]
longitude          n=72    min=25.028603333333333 median=28.525369166666668 max=35.909866666666666 top=[(25.322666666666667, 2), (27.470965, 1), (28.669193333333332, 1), (27.432228333333335, 1)]
sog                n=72    min=0.0       median=0.1       max=16.3      top=[(0.0, 27), (0.1, 11), (0.2, 3), (0.7, 2)]
cog                n=72    min=0.0       median=227.3     max=360.0     top=[(360.0, 11), (281.0, 2), (336.8, 2), (91.6, 1)]
true_heading       n=72    min=9         median=255.0     max=511       top=[(511, 20), (125, 2), (291, 2), (298, 2)]
nav_status         n=72    min=0         median=0.0       max=15        top=[(0, 47), (1, 9), (5, 9), (15, 3)]
rate_of_turn       n=72    min=-128      median=0.0       max=127       top=[(0, 41), (-128, 20), (25, 3), (2, 1)]

wave_height        n=960   min=0.0

## 4. Text and categories: distinct values with `repr()` (family 5)

In [5]:
names = [r for r in con.execute("SELECT vessel_key, mmsi, ship_name FROM vessel ORDER BY vessel_key")]
print("distinct ship names:", len({n for _, _, n in names}), "of", len(names), "vessels")
odd = [(k, m, repr(n), [hex(ord(ch)) for ch in (n or "") if ord(ch) > 127 or ch in "@\u00a0"]) for k, m, n in names
       if n is None or n != n.strip() or n != unicodedata.normalize("NFC", n) or any(ord(ch) > 127 or ch in "@\u00a0" for ch in n)
       or "  " in n or n.strip() == ""]
print("names with NULL, padding, non-ASCII, '@', NBSP or double spaces:", len(odd))
for r in odd[:20]: print("  ", r)
print("\nmost frequent names:", Counter(n for _, _, n in names).most_common(5))
print("\nMMSI length / first digit:")
show("SELECT length(CAST(mmsi AS TEXT)) AS digits, substr(CAST(mmsi AS TEXT), 1, 1) AS first_digit, COUNT(*) AS vessels FROM vessel GROUP BY 1, 2 ORDER BY 1, 2")
print("\nnav_status values:")
show("SELECT nav_status, COUNT(*) AS n FROM ais_position GROUP BY nav_status ORDER BY nav_status")

distinct ship names: 67 of 67 vessels
names with NULL, padding, non-ASCII, '@', NBSP or double spaces: 0

most frequent names: [('AQUILA2', 1), ('AKTI A', 1), ('ISABELL', 1), ('SUPERSTAR', 1), ('AG.ANTONIOS', 1)]

MMSI length / first digit:
digits  first_digit  vessels
------  -----------  -------
9       2            31     
9       3            14     
9       4            1      
9       5            9      
9       6            12     

nav_status values:
nav_status  n 
----------  --
0           47
1           9 
5           9 
8           2 
9           2 
15          3 


## 5. Identity (family 6): same name under two MMSI, first/last time per entity

In [6]:
_TR = str.maketrans({"\u0130": "i", "\u0131": "i", "I": "i", "\u015e": "s", "\u015f": "s", "\u011e": "g", "\u011f": "g", "\u00dc": "u", "\u00fc": "u", "\u00d6": "o", "\u00f6": "o", "\u00c7": "c", "\u00e7": "c"})
def key(n): return " ".join(unicodedata.normalize("NFC", n).replace("@", " ").split()).translate(_TR).casefold()
groups = defaultdict(list)
for k, m, n in names:
    if n and key(n): groups[key(n)].append(m)
dups = {k: v for k, v in groups.items() if len(v) > 1}
print("normalised names shared by more than one MMSI:", dups or "none (check run: names grouped by name key)")
print("\nfirst / last position per vessel (vessels with the earliest and latest last-seen):")
show("SELECT v.mmsi, MIN(p.observed_at) AS first_seen, MAX(p.observed_at) AS last_seen, COUNT(*) AS n FROM ais_position p JOIN vessel v USING (vessel_key) "
     "GROUP BY p.vessel_key ORDER BY n DESC, first_seen LIMIT 10")
print("\nsame MMSI under two names?")
show("SELECT COUNT(*) AS vessels_with_two_names FROM (SELECT mmsi FROM vessel GROUP BY mmsi HAVING COUNT(DISTINCT ship_name) > 1)")

normalised names shared by more than one MMSI: none (check run: names grouped by name key)

first / last position per vessel (vessels with the earliest and latest last-seen):
mmsi       first_seen                last_seen                 n
---------  ------------------------  ------------------------  -
538072248  2026-10-10T11:48:31.013Z  2026-10-10T11:56:50.799Z  2
538008158  2026-10-10T11:48:48.494Z  2026-10-10T11:56:48.370Z  2
667002547  2026-10-10T11:56:31.312Z  2026-10-22T11:37:50.508Z  2
511101443  2026-10-10T11:56:58.587Z  2026-10-22T11:37:55.386Z  2
271044258  2026-10-21T16:12:00.288Z  2026-10-22T11:37:44.335Z  2
239775900  2026-10-10T11:48:26.180Z  2026-10-10T11:48:26.180Z  1
563281500  2026-10-10T11:48:29.402Z  2026-10-10T11:48:29.402Z  1
271000930  2026-10-10T11:48:29.429Z  2026-10-10T11:48:29.429Z  1
341379000  2026-10-10T11:48:30.256Z  2026-10-10T11:48:30.256Z  1
237001000  2026-10-10T11:48:31.013Z  2026-10-10T11:48:31.013Z  1

same MMSI under two names?
vessels_with_two_

## 6. Sequences (family 4): gaps, stuck runs, jumps per marine point; implied speed per vessel

In [7]:
HOUR = 3600
def ts(t): return datetime.strptime(t[:16], "%Y-%m-%dT%H:%M")
for pk, in con.execute("SELECT point_key FROM marine_point ORDER BY point_key").fetchall():
    rows = con.execute("SELECT observed_at, wave_height FROM wave_reading WHERE point_key = ? ORDER BY observed_at", (pk,)).fetchall()
    gaps = sum(1 for a, b in zip(rows, rows[1:]) if (ts(b[0]) - ts(a[0])).total_seconds() != HOUR)
    run = best = 1; bestv = None
    for a, b in zip(rows, rows[1:]):
        if a[1] is not None and a[1] == b[1]: run += 1
        else: run = 1
        if run > best: best, bestv = run, a[1]
    jumps = [abs(b[1] - a[1]) for a, b in zip(rows, rows[1:]) if a[1] is not None and b[1] is not None]
    print("point %s: %d rows, %d non-hourly gaps, longest identical run %d (value %s), largest hourly jump %s m" % (pk, len(rows), gaps, best, bestv, max(jumps) if jumps else None))
def hav(la1, lo1, la2, lo2):
    p = math.pi / 180; a = math.sin((la2-la1)*p/2)**2 + math.cos(la1*p)*math.cos(la2*p)*math.sin((lo2-lo1)*p/2)**2
    return 2 * 3440.065 * math.asin(math.sqrt(a))
multi = con.execute("SELECT vessel_key FROM ais_position GROUP BY vessel_key HAVING COUNT(*) > 1").fetchall()
print("\nvessels with 2+ positions (needed for implied-speed checks):", len(multi))
for (vk,) in multi[:10]:
    r = con.execute("SELECT observed_at, latitude, longitude, sog FROM ais_position WHERE vessel_key = ? AND latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180 ORDER BY observed_at", (vk,)).fetchall()
    for a, b in zip(r, r[1:]):
        dt = (datetime.strptime(b[0][:19], "%Y-%m-%dT%H:%M:%S") - datetime.strptime(a[0][:19], "%Y-%m-%dT%H:%M:%S")).total_seconds() / 3600
        if dt > 0: print("  vessel_key %s: implied %.1f kn over %.2f h, reported sog %s -> %s" % (vk, hav(a[1], a[2], b[1], b[2]) / dt, dt, a[3], b[3]))

point 1: 120 rows, 1 non-hourly gaps, longest identical run 24 (value 0.02), largest hourly jump 0.07999999999999999 m
point 2: 120 rows, 1 non-hourly gaps, longest identical run 15 (value 0.04), largest hourly jump 0.08000000000000002 m
point 3: 120 rows, 1 non-hourly gaps, longest identical run 97 (value 0.02), largest hourly jump 0.020000000000000004 m
point 4: 120 rows, 1 non-hourly gaps, longest identical run 84 (value 0.02), largest hourly jump 0.020000000000000018 m
point 5: 120 rows, 1 non-hourly gaps, longest identical run 21 (value 0.02), largest hourly jump 0.039999999999999994 m
point 6: 120 rows, 1 non-hourly gaps, longest identical run 52 (value 0.02), largest hourly jump 0.06 m
point 7: 120 rows, 1 non-hourly gaps, longest identical run 24 (value 0.02), largest hourly jump 0.12000000000000001 m
point 8: 120 rows, 1 non-hourly gaps, longest identical run 9 (value 0.32), largest hourly jump 0.32 m

vessels with 2+ positions (needed for implied-speed checks): 5
  vessel_key

## 7. Comparing fields (family 4) and robust outlier candidates (family 3)

In [8]:
print("sog by nav_status (a vessel 'at anchor' or 'moored' should not be doing many knots):")
show("SELECT nav_status, COUNT(*) AS n, ROUND(MIN(sog), 1) AS min_sog, ROUND(AVG(sog), 2) AS mean_sog, MAX(sog) AS max_sog FROM ais_position WHERE sog < 102.3 GROUP BY nav_status ORDER BY nav_status")
print("\nwind-wave height above total wave height:")
show("SELECT COUNT(*) AS rows_checked, SUM(wind_wave_height > wave_height + 0.05) AS wind_over_total FROM wave_reading WHERE wave_height IS NOT NULL AND wind_wave_height IS NOT NULL")
sog = sorted(v for (v,) in con.execute("SELECT sog FROM ais_position WHERE sog IS NOT NULL AND sog < 102.3"))
if len(sog) >= 4:
    q1, _, q3 = statistics.quantiles(sog, n=4, method="inclusive")
    fence = q3 + 1.5 * (q3 - q1)
    print("\nsog IQR fence: Q1=%s Q3=%s upper fence=%.2f -> values above:" % (q1, q3, fence), [v for v in sog if v > fence])
print("\nwave_height per point: median, MAD and robust z above 3.5 (kept and flagged in M3, never deleted):")
for pk, in con.execute("SELECT DISTINCT point_key FROM wave_reading WHERE wave_height IS NOT NULL ORDER BY point_key").fetchall():
    v = [x for (x,) in con.execute("SELECT wave_height FROM wave_reading WHERE point_key = ? AND wave_height BETWEEN 0 AND 25", (pk,))]
    med = statistics.median(v); mad = statistics.median(abs(x - med) for x in v)
    hi = [x for x in v if mad > 0 and 0.6745 * (x - med) / mad > 3.5]
    print("  point %s: n=%d median=%.2f MAD=%.3f max=%.2f flagged=%d" % (pk, len(v), med, mad, max(v), len(hi)))
print("\nhours with wave_height above 1.5 m (the project question needs these):")
show("SELECT COUNT(*) AS rows_over_1_5, COUNT(DISTINCT point_key) AS points FROM wave_reading WHERE wave_height > 1.5")

sog by nav_status (a vessel 'at anchor' or 'moored' should not be doing many knots):
nav_status  n   min_sog  mean_sog  max_sog
----------  --  -------  --------  -------
0           47  0.0      5.34      16.3   
1           9   0.0      0.02      0.1    
5           9   0.0      0.89      8.0    
8           2   0.0      0.55      1.1    
9           2   0.0      0.0       0.0    
15          3   0.0      0.27      0.7    

wind-wave height above total wave height:
rows_checked  wind_over_total
------------  ---------------
960           0              

sog IQR fence: Q1=0.0 Q3=9.325000000000001 upper fence=23.31 -> values above: []

wave_height per point: median, MAD and robust z above 3.5 (kept and flagged in M3, never deleted):
  point 1: n=120 median=0.07 MAD=0.050 max=0.50 flagged=17
  point 2: n=120 median=0.08 MAD=0.060 max=0.64 flagged=12
  point 3: n=120 median=0.02 MAD=0.000 max=0.08 flagged=0
  point 4: n=120 median=0.02 MAD=0.000 max=0.14 flagged=0
  point 5: n=120 media